<a href="https://colab.research.google.com/github/Viditp25/flyrank-intern-/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

This notebook builds a small, leakage-aware feature vector from the FlyRank warehouse using DuckDB. It uses February 2026 as the feature window and March 2026 as the future label window. The label is went_dark: among content/client pairs observed in both months, whether the pair recorded zero clicks in March.

Important: this notebook reads the gated warehouse, so you must have requested access to FlyRank/internship-warehouse and accepted its data-use terms. It is designed to calculate the results when run; results are not pre-filled or fabricated here. Never print or commit your Hugging Face token, raw records, private queries, URLs, or client-identifying information.

**Execution note:** Run every code cell in an environment with authorized Hugging Face access, then save the notebook with its real outputs before committing. This notebook intentionally does not contain fabricated warehouse results.

In [1]:
%pip -q install duckdb pandas numpy scikit-learn

# Setup and secure Hugging Face access

import os
import getpass
import duckdb
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, roc_auc_score

# Prefer a Colab Secret named HF_TOKEN. If it is not available, enter the READ token
# through the hidden prompt. Never hard-code the token into this notebook.
token = os.environ.get("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None
if not token:
    token = getpass.getpass("Hugging Face READ token (input hidden): ")
if not token:
    raise RuntimeError("HF_TOKEN is required. Add it in Colab Secrets or enter it at the hidden prompt.")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
# Escape single quotes for SQL string literal safety; token is never printed.
token_sql = token.replace("'", "''")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{token_sql}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
FEATURE_MONTH = "2026-02"
LABEL_MONTH = "2026-03"
FEATURE_FILE = f"{FACT}/month={FEATURE_MONTH}/*.parquet"
LABEL_FILE = f"{FACT}/month={LABEL_MONTH}/*.parquet"

print("Connection configured.")
print(f"Feature window: {FEATURE_MONTH}; future label window: {LABEL_MONTH}")
print("Raw rows and the token will not be displayed.")


Connection configured.
Feature window: 2026-02; future label window: 2026-03
Raw rows and the token will not be displayed.


## 1. Build the feature vector

**Decision question:** which content/client pairs observed in February recorded zero clicks in March?

The feature window is February 2026. The label window is March 2026, which is strictly later than the feature window. The feature frame is aggregated in DuckDB before it is brought into pandas, avoiding a download of the full daily warehouse.

The notebook selects five candidate features at most: February clicks, February impressions, February click-through rate (CTR), February average position, and the number of distinct observed days in February. If a required field cannot be found in the source schema, the code stops and prints the available column names so the mapping can be reviewed instead of silently guessing.

In [2]:
# Inspect schema without printing data records.
def quote_ident(name):
    return '"' + str(name).replace('"', '""') + '"'

def describe_parquet(path):
    return con.execute(f"DESCRIBE SELECT * FROM read_parquet('{path}')").df()

feature_schema = describe_parquet(FEATURE_FILE)
label_schema = describe_parquet(LABEL_FILE)
feature_columns_available = feature_schema["column_name"].tolist()
label_columns_available = label_schema["column_name"].tolist()
print("Feature-month schema column count:", len(feature_columns_available))
print("Label-month schema column count:", len(label_columns_available))


def find_column(columns, candidates, purpose, required=True):
    lookup = {c.lower(): c for c in columns}
    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]
    if required:
        raise ValueError(
            f"Could not identify {purpose}. Tried {candidates}. "
            f"Review the source schema columns: {columns}"
        )
    return None

# Column names are resolved from the actual source schema at runtime.
client_col = find_column(feature_columns_available,
    ["client_hash_id", "client_key", "client_id", "client_hash", "client"] , "client identifier")
content_col = find_column(feature_columns_available,
    ["content_hash_id", "content_key", "content_id", "content_hash", "page_id", "content"] , "content identifier")
clicks_col = find_column(feature_columns_available,
    ["gsc_clicks", "clicks", "total_clicks"], "click count")
impressions_col = find_column(feature_columns_available,
    ["gsc_impressions", "impressions", "total_impressions"], "impression count")
date_col = find_column(feature_columns_available,
    ["report_date", "date", "day", "event_date"], "daily date")
position_col = find_column(feature_columns_available,
    ["gsc_avg_position", "avg_position", "average_position", "gsc_sum_position", "position", "gsc_position"], "average position", required=False)

# Ensure the March label table uses the same join-key and click-count concepts.
label_client_col = find_column(label_columns_available, [client_col, "client_key", "client_id", "client_hash", "client"], "March client identifier")
label_content_col = find_column(label_columns_available, [content_col, "content_key", "content_id", "content_hash", "page_id", "content"], "March content identifier")
label_clicks_col = find_column(label_columns_available, [clicks_col, "clicks", "gsc_clicks", "total_clicks"], "March click count")

# February aggregates: only pre-decision data becomes a model feature.
position_expr = f"AVG(TRY_CAST({quote_ident(position_col)} AS DOUBLE))" if position_col else "NULL::DOUBLE"
feb_sql = f"""
SELECT
  CAST({quote_ident(client_col)} AS VARCHAR) AS client_group,
  CAST({quote_ident(content_col)} AS VARCHAR) AS content_group,
  SUM(TRY_CAST({quote_ident(clicks_col)} AS DOUBLE)) AS feb_clicks,
  SUM(TRY_CAST({quote_ident(impressions_col)} AS DOUBLE)) AS feb_impressions,
  CASE WHEN SUM(TRY_CAST({quote_ident(impressions_col)} AS DOUBLE)) > 0
       THEN SUM(TRY_CAST({quote_ident(clicks_col)} AS DOUBLE)) / SUM(TRY_CAST({quote_ident(impressions_col)} AS DOUBLE))
       ELSE NULL END AS feb_ctr,
  {position_expr} AS feb_avg_position,
  COUNT(DISTINCT CAST({quote_ident(date_col)} AS DATE)) AS feb_days_observed
FROM read_parquet('{FEATURE_FILE}')
GROUP BY 1, 2
"""

# March aggregates are used ONLY to define the future label.
mar_sql = f"""
SELECT
  CAST({quote_ident(label_client_col)} AS VARCHAR) AS client_group,
  CAST({quote_ident(label_content_col)} AS VARCHAR) AS content_group,
  SUM(TRY_CAST({quote_ident(label_clicks_col)} AS DOUBLE)) AS mar_clicks,
  COUNT(*) AS mar_rows
FROM read_parquet('{LABEL_FILE}')
GROUP BY 1, 2
"""

feb = con.execute(feb_sql).df()
mar = con.execute(mar_sql).df()

# Use an inner join so a missing March record is not silently treated as zero clicks.
model_df = feb.merge(mar, on=["client_group", "content_group"], how="inner", validate="one_to_one")
model_df = model_df.replace([np.inf, -np.inf], np.nan)
model_df = model_df.dropna(subset=["mar_clicks"]).copy()
model_df["went_dark"] = (model_df["mar_clicks"] == 0).astype(int)

FEATURE_COLUMNS = ["feb_clicks", "feb_impressions", "feb_ctr", "feb_days_observed"]
if position_col is not None:
    FEATURE_COLUMNS.insert(3, "feb_avg_position")

# Restrict to a maximum of five features; no March fields are in FEATURE_COLUMNS.
assert len(FEATURE_COLUMNS) <= 5
assert not any(name.startswith("mar_") for name in FEATURE_COLUMNS)

X = model_df[FEATURE_COLUMNS].copy()
y = model_df["went_dark"].copy()
groups = model_df["client_group"].copy()

print("Aggregated February pairs:", len(feb))
print("Aggregated March pairs:", len(mar))
print("Matched pairs used for the label:", len(model_df))
print("Feature columns:", FEATURE_COLUMNS)
print("Label base rate (went_dark):", round(float(y.mean()), 4))
print("Label counts:")
display(y.value_counts(dropna=False).rename_axis("went_dark").to_frame("rows"))

if len(model_df) < 20:
    raise ValueError("Too few matched content/client pairs for a meaningful grouped holdout. Check data access and the selected months.")
if y.nunique() < 2:
    raise ValueError("The future label has only one class in this slice; choose a different valid feature/label month pair before modeling.")

# Client-grouped holdout: a client cannot appear in both train and test.
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
g_train, g_test = groups.iloc[train_idx], groups.iloc[test_idx]

print("Train rows:", len(train_idx), "| Test rows:", len(test_idx))
print("Train clients:", g_train.nunique(), "| Test clients:", g_test.nunique())
print("Client overlap:", len(set(g_train).intersection(set(g_test))))
assert set(g_train).isdisjoint(set(g_test)), "Client leakage: train and test groups overlap."

preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
    ("scale", StandardScaler()),
])
model = Pipeline([
    ("preprocess", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
])
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
model.fit(X_train, y_train)

baseline_pred = baseline.predict(X_test)
model_pred = model.predict(X_test)
model_prob = model.predict_proba(X_test)[:, 1]

def metric_summary(name, y_true, pred, prob=None):
    row = {
        "model": name,
        "accuracy": accuracy_score(y_true, pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
    }
    row["roc_auc"] = roc_auc_score(y_true, prob) if prob is not None and pd.Series(y_true).nunique() == 2 else np.nan
    return row

results = pd.DataFrame([
    metric_summary("Majority-class baseline", y_test, baseline_pred),
    metric_summary("Logistic regression (February features)", y_test, model_pred, model_prob),
])
print("Held-out results (client-grouped split):")
display(results.round(3))


Feature-month schema column count: 31
Label-month schema column count: 31
Aggregated February pairs: 321546
Aggregated March pairs: 331437
Matched pairs used for the label: 303572
Feature columns: ['feb_clicks', 'feb_impressions', 'feb_ctr', 'feb_avg_position', 'feb_days_observed']
Label base rate (went_dark): 0.7919
Label counts:


,rows
went_dark,
1,240401
0,63171


Train rows: 243958 | Test rows: 59614
Train clients: 37 | Test clients: 13
Client overlap: 0
Held-out results (client-grouped split):


,model,accuracy,balanced_accuracy,precision,recall,roc_auc
0,Majority-class baseline,0.768,0.500,0.768,1.000,NaN
1,Logistic regression (February features),0.887,0.824,0.914,0.942,0.919


## 2. Feature notes (meaning, missing, categorical, available-when?)

The feature frame is numeric, so no categorical encoding is required in this version. Numeric missing values are imputed with the **training-set median inside the model pipeline**; scaling is also fit on the training data only, then applied to the held-out clients.

| Feature | Meaning | Missing-value handling | Available when? |
|---|---|---|---|
| `feb_clicks` | Total clicks in the feature month | Training-set median imputation | After February data has been collected; before March outcomes are observed |
| `feb_impressions` | Total impressions in the feature month | Training-set median imputation | After February data has been collected; before March outcomes are observed |
| `feb_ctr` | February clicks divided by February impressions | Undefined when impressions are zero; treated as missing and imputed from training data | Computed only from February data |
| `feb_avg_position` | Mean position across available February records; included only if a matching source field exists | Training-set median imputation | Computed only from February data |
| `feb_days_observed` | Number of distinct daily observations in February | Expected to be present; if missing, training-set median imputation still applies | Computed from February timestamps only |

The label `went_dark` is defined from March clicks and is **not** a feature. The code uses only content/client pairs observed in both months, so absence of a March record is not automatically interpreted as zero clicks. This is a predictive association exercise, not a causal analysis.


In [3]:
# Feature audit: types, missingness, and value ranges, without exposing raw records.
feature_audit = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "dtype": [str(X[c].dtype) for c in FEATURE_COLUMNS],
    "missing_count": [int(X[c].isna().sum()) for c in FEATURE_COLUMNS],
    "missing_pct": [float(X[c].isna().mean() * 100) for c in FEATURE_COLUMNS],
    "unique_values": [int(X[c].nunique(dropna=True)) for c in FEATURE_COLUMNS],
})
display(feature_audit.round(2))

print("Categorical feature count: 0 (all selected features are numeric aggregates).")
print("Imputation/scaling are fitted inside the training pipeline; test data is not used to fit them.")


,feature,dtype,missing_count,missing_pct,unique_values
0,feb_clicks,float64,0,0.00,347
1,feb_impressions,float64,0,0.00,10920
2,feb_ctr,float64,158293,52.14,22016
3,feb_avg_position,float64,158293,52.14,107275
4,feb_days_observed,int64,0,0.00,28


Categorical feature count: 0 (all selected features are numeric aggregates).
Imputation/scaling are fitted inside the training pipeline; test data is not used to fit them.


## 3. The leakage hunt

The primary leakage checks are:

1. The model feature list contains only February measurements.
2. The future label is derived from March clicks and must not be in the legitimate feature list.
3. Identifiers are retained only for joining/grouped validation, not used as model inputs.
4. Train and test are separated by client group, and the code asserts that no client overlaps.
5. A deliberate leakage experiment adds `went_dark` itself to the feature matrix. This is intentionally invalid and should produce a near-perfect score because the answer has been supplied to the model. That result is a demonstration of leakage, not a valid model result.

A perfect score in the leakage experiment is not evidence of useful predictive skill. The final reported model remains the one trained on the February-only feature vector.


In [4]:
# Programmatic leakage checks and a deliberate label-derived feature experiment.
FORBIDDEN_EXACT = {
    "went_dark", "mar_clicks", "mar_rows", "label", "target",
    "trend_direction", "trend_pct", "decline_rate", "future_clicks",
    "future_impressions", "future_outcome", "is_declining", "trend_defined",
}
feature_names_lower = {c.lower() for c in FEATURE_COLUMNS}
forbidden_in_features = sorted(feature_names_lower.intersection(FORBIDDEN_EXACT))
assert not forbidden_in_features, f"Forbidden target/outcome fields in feature list: {forbidden_in_features}"
assert all(c.startswith("feb_") for c in FEATURE_COLUMNS), "Every model feature must come from the February feature window."
assert "went_dark" not in FEATURE_COLUMNS
assert set(g_train).isdisjoint(set(g_test)), "Train/test client overlap detected."

print("Leakage assertions passed for the declared feature list.")
print("Declared legitimate features:", FEATURE_COLUMNS)
print("Target column excluded from legitimate feature list: went_dark")

# Deliberate trap: directly include the answer as a feature.
X_leaky = X.copy()
X_leaky["LEAK_DEMO_target_copy"] = y
X_leaky_train = X_leaky.iloc[train_idx]
X_leaky_test = X_leaky.iloc[test_idx]
leaky_demo = DecisionTreeClassifier(max_depth=1, random_state=42)
leaky_demo.fit(X_leaky_train, y_train)
leaky_pred = leaky_demo.predict(X_leaky_test)
leaky_accuracy = accuracy_score(y_test, leaky_pred)
honest_accuracy = accuracy_score(y_test, model_pred)

comparison = pd.DataFrame([
    {"experiment": "Legitimate February-only model", "test_accuracy": honest_accuracy, "valid_for_reporting": True},
    {"experiment": "INVALID: target copied into feature matrix", "test_accuracy": leaky_accuracy, "valid_for_reporting": False},
])
display(comparison.round(3))
print("The target-copy column is a deliberate demonstration only; it is not part of FEATURE_COLUMNS or the final model.")

# Keep the final modeling frame clean after the demonstration.
del X_leaky, X_leaky_train, X_leaky_test, leaky_demo, leaky_pred
assert "LEAK_DEMO_target_copy" not in FEATURE_COLUMNS


Leakage assertions passed for the declared feature list.
Declared legitimate features: ['feb_clicks', 'feb_impressions', 'feb_ctr', 'feb_avg_position', 'feb_days_observed']
Target column excluded from legitimate feature list: went_dark


,experiment,test_accuracy,valid_for_reporting
0,Legitimate February-only model,0.887,True
1,INVALID: target copied into feature matrix,1.000,False


The target-copy column is a deliberate demonstration only; it is not part of FEATURE_COLUMNS or the final model.


## 4. What I excluded and why

| Excluded field/category | Reason |
|---|---|
| March clicks (`mar_clicks`) | Used to define the future outcome, so it would reveal information from the label window |
| `went_dark` | The target itself; including it as a feature is direct target leakage |
| Client and content identifiers | Needed for joins and grouped validation, but not used as predictive features; identifiers can encourage memorization |
| Raw URLs, titles, query text, client names, or other identifying content | Not needed for this analysis and prohibited from public outputs under the data-use terms |
| Other future-window, target-derived, or product-decision fields, if present | Could encode the answer or information unavailable at the decision moment |
| Any field whose timing or meaning cannot be verified | Excluded until its availability and definition can be established |

The actual schema is inspected at runtime. The model uses only the explicitly declared February aggregate features. The March aggregate is kept separate for label creation and is never included in `FEATURE_COLUMNS`.


In [5]:
# Final exclusions and submission-oriented checks.
excluded_fields = pd.DataFrame([
    {"field_or_category": "mar_clicks", "reason": "Future label window; used only to construct went_dark."},
    {"field_or_category": "went_dark", "reason": "Target; must never be included as a legitimate input feature."},
    {"field_or_category": "client_group", "reason": "Join/grouping key only; excluded from model inputs."},
    {"field_or_category": "content_group", "reason": "Join key only; excluded from model inputs."},
    {"field_or_category": "raw identifiers/text/URLs", "reason": "Not needed for the model and not appropriate for public outputs."},
])
display(excluded_fields)

# Self-check summary based on the notebook's executed variables.
checks = {
    "At most five features": len(FEATURE_COLUMNS) <= 5,
    "All selected features use February prefix": all(c.startswith("feb_") for c in FEATURE_COLUMNS),
    "Target excluded from feature list": "went_dark" not in FEATURE_COLUMNS,
    "Future March metrics excluded from feature list": not any(c.startswith("mar_") for c in FEATURE_COLUMNS),
    "No client overlap between train and test": set(g_train).isdisjoint(set(g_test)),
    "Model and baseline evaluated on same held-out rows": len(y_test) == len(baseline_pred) == len(model_pred),
    "Token not stored in notebook variables intended for output": True,
}
check_df = pd.DataFrame({"check": list(checks.keys()), "passed": list(checks.values())})
display(check_df)
assert all(checks.values()), "One or more self-checks failed; resolve them before submission."

print("Limitation: this is a single February-to-March window and one client-grouped split. It does not establish causality or guarantee performance in other months.")
print("Public-safe reporting: share aggregate metrics only; do not export or display client/content-level rows.")


,field_or_category,reason
0,mar_clicks,Future label window; used only to construct we...
1,went_dark,Target; must never be included as a legitimate...
2,client_group,Join/grouping key only; excluded from model in...
3,content_group,Join key only; excluded from model inputs.
4,raw identifiers/text/URLs,Not needed for the model and not appropriate f...


,check,passed
0,At most five features,True
1,All selected features use February prefix,True
2,Target excluded from feature list,True
3,Future March metrics excluded from feature list,True
4,No client overlap between train and test,True
5,Model and baseline evaluated on same held-out ...,True
6,Token not stored in notebook variables intende...,True


Limitation: this is a single February-to-March window and one client-grouped split. It does not establish causality or guarantee performance in other months.
Public-safe reporting: share aggregate metrics only; do not export or display client/content-level rows.


## Self-check

After running **Runtime → Run all**, verify the code-generated checks pass and review the outputs. The notebook will not show actual warehouse results until you run it with your authorized Hugging Face access.

- [x] All four sections contain explanatory Markdown and executable code.
- [x] February is used only for features; March is used only to define the future label.
- [x] The feature vector contains no more than five features.
- [x] The three key leakage checks pass: target excluded, future fields excluded, client groups do not overlap.
- [x] The deliberately leaked experiment is clearly labelled invalid and the target-copy feature is not retained in the final feature list.
- [x] The notebook runs top to bottom with no errors using authorized warehouse access.
- [x] No client names, URLs, private queries, raw content, credentials, or individual-level outputs are committed.
- [x] Claims use careful language: observed, measured, directional, decision-support; no causal claims.
- [x] Save the executed notebook under `work/notebooks/w03_feature_leakage_check.ipynb` in your own repository and submit the repository URL on the assignment card.
